In [0]:
from pyspark.sql.functions import sum, count, avg, desc

silver_df = spark.read.table("workspace.default.silver_retail_sales")

product_sales = silver_df.groupBy("Product") \
    .agg(
        sum("Revenue").alias("Total_Revenue"),
        sum("Quantity").alias("Total_Quantity"),
        count("Order_ID").alias("Total_Orders")
    )

product_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_product_sales")
city_sales = silver_df.groupBy("City") \
    .agg(
        sum("Revenue").alias("Total_Revenue"),
        count("Order_ID").alias("Total_Orders")
    ) \
    .orderBy(desc("Total_Revenue"))

display(city_sales)
city_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_city_sales")
sales_summary = silver_df.agg(
    sum("Revenue").alias("Total_Revenue"),
    count("Order_ID").alias("Total_Orders"),
    avg("Revenue").alias("Average_Order_Value")
)

display(sales_summary)
sales_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_sales_summary")
display(spark.sql("SHOW TABLES IN workspace.default"))

City,Total_Revenue,Total_Orders
Mumbai,62000,2
Pune,59500,2
Bangalore,55000,1
Unknown,1600,1
Delhi,1600,1


Total_Revenue,Total_Orders,Average_Order_Value
179700,7,25671.428571428572


database,tableName,isTemporary
default,bronze_retail_sales,false
default,gold_city_sales,false
default,gold_product_sales,false
default,gold_sales_summary,false
default,silver_retail_sales,false
